get all relevant sensititre and vitek files

In [1]:
import pandas as pd
import os


df_vitek = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/bin/VITEK_combined_02_26_interpreted_bin.csv"
)
df_sensititre = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/bin/Sensititre_combined_02_26_interpreted_bin.csv"
)

available_ids_vitek = {
    os.path.splitext(f)[0]
    for f in os.listdir("/groups/ds/Win-KID/BVBRC/VITEK/gff_card_401")
    if f.endswith(".gff")
}
available_ids_sensititre = {
    os.path.splitext(f)[0]
    for f in os.listdir("/groups/ds/Win-KID/BVBRC/sensititre/gff_card_401")
    if f.endswith(".gff")
}

# Remove unavialible IDs
df_vitek = df_vitek[
    df_vitek["Sample_ID_IfH"].astype(str).isin(available_ids_vitek)
].reset_index(drop=True)
df_sensititre = df_sensititre[
    df_sensititre["Sample_ID_IfH"].astype(str).isin(available_ids_sensititre)
].reset_index(drop=True)

# Remove non ECO samples
df_vitek = df_vitek[df_vitek["Organism_Code"] == "ECO"]
df_sensititre = df_sensititre[df_sensititre["Organism_Code"] == "ECO"]

df_vitek = df_vitek.dropna(subset=df_vitek.columns[2:], how="all")
df_sensititre = df_sensititre.dropna(subset=df_sensititre.columns[2:], how="all")

df_vitek.to_csv("/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/vitek.csv", index=False)
df_sensititre.to_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/sensititre.csv", index=False
)

print(len(df_vitek))
print(df_vitek.notna().sum(axis=1).mean())
print(len(df_sensititre))
print(df_sensititre.notna().sum(axis=1).mean())

1342
10.863636363636363
221
14.53393665158371


Cross validate EKP NA values
Random select EKP Phoenix values 
Random select EKP VITEK values CV

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Clean up vitek
df_vitek = pd.read_csv("/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/vitek.csv")
df_vitek_cleanup = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/samples_used_vitek_10.csv"
)
df_vitek_cleaned = df_vitek[
    df_vitek["Sample_ID_IfH"].isin(df_vitek_cleanup["Sample_ID_IfH"])
]

df_vitek_cleaned.to_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/vitek_cleaned.csv", index=False
)

# Clean up sensititre
df_sensititre = pd.read_csv("/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/sensititre.csv")
df_sensisitre_cleanup = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/samples_used_sensititre_10.csv"
)
df_sensititre_cleaned = df_sensititre[
    df_sensititre["Sample_ID_IfH"].isin(df_sensisitre_cleanup["Sample_ID_IfH"])
]

df_sensititre_cleaned.to_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/sensititre_cleaned.csv", index=False
)

Create and save splits

In [11]:
import pandas as pd
from sklearn.model_selection import KFold
from pathlib import Path

N_REPEATS = 1
N_SPLITS = 4
OUTPUT_DIR = Path("/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV")

df_vitek = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/vitek_cleaned.csv"
)
df_sensititre = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/sensititre_cleaned.csv"
)

remainder = len(df_sensititre) % N_SPLITS
if remainder > 0:
    # zufällig Zeilen auswählen, die entfernt werden
    df_sensititre = df_sensititre.sample(
        n=len(df_sensititre) - remainder, random_state=42
    ).reset_index(drop=True)


for repeat in range(N_REPEATS):
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=repeat)

    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(df_sensititre)):
        seed = repeat * 100 + fold_idx
        run_name = f"repeat{repeat}_fold{fold_idx}"

        run_dir = OUTPUT_DIR / run_name
        run_dir.mkdir(parents=True, exist_ok=True)

        # -----------------------------
        # NA SPLIT
        # -----------------------------
        na_train = df_sensititre.iloc[test_idx]  # Change test and train
        na_test = df_sensititre.iloc[train_idx]  # Change test and train

        # -----------------------------
        # SAVE
        # -----------------------------
        na_train.to_csv(run_dir / "sensititre_train.csv", index=False)
        na_test.to_csv(run_dir / "sensititre_test.csv", index=False)

        print(f"Saved: {run_dir}")

Saved: /groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/repeat0_fold0
Saved: /groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/repeat0_fold1
Saved: /groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/repeat0_fold2
Saved: /groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/repeat0_fold3
